# 00. 환경 및 데이터 확인

실습을 시작하기 전에 패키지와 데이터가 제대로 준비되었는지 확인합니다.

**연구 질문** — IFN-beta 자극을 받은 PBMC 는 세포 타입별로 어떻게 다르게 반응하는가?

In [ ]:
import scanpy as sc
import pandas as pd

sc.settings.verbosity = 1
sc.logging.print_header()

In [ ]:
adata = sc.read_h5ad("../data/processed/gse96583_ifnb.h5ad")
adata

## 실험 설계 확인

대조군(`ctrl`)과 IFN-beta 자극군(`stim`) 이 공여자별로 어떻게 분포하는지 봅니다.

> 세포 타입 주석은 들어 있지 않습니다. 클러스터링과 마커 유전자로 **직접 붙이는 것이 실습 과제**입니다.

In [ ]:
pd.crosstab(adata.obs["donor"], adata.obs["stim"], margins=True)

In [ ]:
# 사용 가능한 metadata 확인 (세포 타입 라벨은 포함되어 있지 않습니다)
adata.obs.head()

## raw count 인지 확인

`.X` 는 정규화되지 않은 정수 count 여야 합니다. QC·정규화부터는 직접 수행합니다.

In [ ]:
print("최댓값:", adata.X.max())
print("정수 여부:", (adata.X.data % 1 == 0).all())

# 참고: MT- 유전자는 목록에 있으나 count 가 전부 0 이라 미토콘드리아 비율 QC 는 사용할 수 없습니다.
adata.obs[["total_counts", "n_genes"]].describe()

## 맛보기: IFN 반응 유전자

대표적인 인터페론 자극 유전자(ISG)인 `ISG15` 의 발현이 자극군에서 올라가는지 확인합니다.
(정규화 전 raw count 기준의 대략적인 확인입니다)

In [ ]:
isg = adata[:, "ISG15"].X.toarray().ravel()
pd.DataFrame({"stim": adata.obs["stim"].values, "ISG15": isg}).groupby("stim", observed=True).mean()